# Homework 2

First I import pyspark to enable me to work with the package.

In [0]:
import pyspark

**1. What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.**

To carry out the above-mentioned calculations, I load the pit stops dataset. I  also inspect the dataset to get an idea of the structure of the data.

In [0]:
df_pitstops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)

In [0]:
display(df_pitstops)

- read.csv enabled me to load the dataset, which is in csv format. I also included "header=True" to ensure that the first row is read as headers.
- the display function enabled me to inspect the entire datasetand get all rows and columns.

To compute the average time (in milliseconds) for each driver, I import the average (avg) function. I also convert the raceId and driverId froom string datatype to integer in order to be able to compute averages in ascending order of the raceId and driverId. It is difficult to do so when the raceId and driverId are strings.


In [0]:
from pyspark.sql.functions import avg

In [0]:
# convert raceId and driverId to int
df_pitstops = df_pitstops.withColumn("raceId", df_pitstops["raceId"].cast("int")).withColumn("driverId", df_pitstops["driverId"].cast("int"))
display(df_pitstops)

In the above code, "withColumn" enabled me to replace raceId and driverId with updated versions while "cast" enabled me to change the datatypes.

In [0]:
# Calculate the average time each driver spent at the pit stop for each race
df_avg = df_pitstops.groupBy("raceId", "driverId").agg(avg("milliseconds")).orderBy("raceId", "driverId")
display(df_avg)

In the above code, "groupBy" enables me to group the data by raceId and driverId and compute the summary statistics for each group. "avg" enables me to compute the average per group, while "orderBy" enables me to sort the final results by raceId, then driverId, so that it is easier to read.

To compute the slowest and fastest pit stop for each race, I import the min and max aggregate functions. To get the slowest and fastest pit stop for each race, I group the data by raceId then calculate the maximum pit stop duration in milliseconds for each race and rename it to "slowest". I also calculate the minimum pit stop duration for each race and rename it to "fastest".

In [0]:
from pyspark.sql.functions import min, max

In [0]:
# Compute the slowest and fastest pit stop durations for each race
max_and_min = df_pitstops.groupBy("raceId").agg(max("milliseconds").alias("slowest"), min("milliseconds").alias("fastest"))
display(max_and_min)

In the above code "groupBy" enabled me to group the data by raceId, while "max" and "min" enabled me to calculate the slowest and fastest time, respectively. "alias" enabled me to rename max("milliseconds") and min("milliseconds") to "slowest" and "fastest", respectively.

**2. Rank order by finishing position the average time spent at the pit stop in each race.**

- To compute the position number of each driver based on the average pit stop time per race, I import the row_number, desc, and Window functions. "row_number" enables me to assign ranks and "Window" to define groups for ranking without collapsing rows.
- After importing the relevant functions, I create a new column called "position", which indicates the rank of each driver for each race. I then split the data into groups or windows by each race, ensuring that the ranking happens within each race and not across all races. I then order the data in ascending order, so that the fastest driver (shortest pit stop time) is position 1, and so on.
- Note: I use df_avg for the computation and not df_pitstops as the former has the relevant aggregate calculations.

In [0]:
# Add column called position (the position of each driver in the race) and rank the drivers by their average pit stop time
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

df_avg = df_avg.withColumn("position", row_number().over(Window.partitionBy("raceId").orderBy("avg(milliseconds)")))
display(df_avg)

- In the above code, "withColumn" enables me to create a new column called "position". 
- "Window.partitionBy("raceId")" enables me to split the data into separate windows by race and ensure that the ranking happens within each race and not across all races. 
- "orderBy" enables me to sort the drivers by average pit stop time for each race. 
- "row_number().over()" enables me to assign a rank after sorting.